In [6]:
"""
Create a symphony programmatically with reusable block definitions.

This example demonstrates how to define a group of assets once as a variable
and reference it in multiple places. This makes it easy to update the same
assets in multiple locations by changing just one variable.
"""

from composer import ComposerClient
from composer.models.common.symphony import (
    Asset,
    WeightCashEqual,
    Group,
    SymphonyDefinition,
    WeightMap,
)
from dotenv import load_dotenv
import json
import os

load_dotenv()

# Initialize client
client = ComposerClient(
    api_key=os.getenv("COMPOSER_API_KEY"),
    api_secret=os.getenv("COMPOSER_API_SECRET"),
)

# =============================================================================
# DEFINE REUSABLE BLOCKS
# =============================================================================
# Here's the key: define your asset group as a variable ONCE, then reference
# it multiple times. When you want to change a ticker, just update here!

# Create a weight block with equal weight across these assets
tech_weight_block = WeightCashEqual(
    children=[
        Asset(ticker="MSFT", name="Microsoft Corporation", exchange="XNAS"),
        Asset(ticker="AAPL", name="Apple Inc", exchange="XNAS"),
        Asset(ticker="GOOG", name="Alphabet Inc Class A", exchange="XNAS"),
    ]
)


# =============================================================================
# USE THE BLOCK MULTIPLE TIMES
# =============================================================================
# Reference the same block multiple times in the symphony structure.
# The structure is: SymphonyDefinition -> WeightCashEqual -> [Group, Group]
# Note: IDs are auto-generated as proper UUIDs if not provided

symphony = SymphonyDefinition(
    name="Tech Block Reference Demo",
    description="Demonstrates using the same block in multiple places",
    rebalance="daily",
    children=[
        # Single WeightCashEqual containing multiple Groups (the reusable blocks)
        WeightCashEqual(
            children=[
                # First reference: Group 1 (same tech_weight_block!)
                Group(
                    name="Tech Group 1",
                    collapsed=False,
                    children=[tech_weight_block],
                ),
                # Second reference: Group 2 (same tech_weight_block!)
                Group(
                    name="Tech Group 2",
                    collapsed=False,
                    children=[tech_weight_block],
                ),
            ]
        ),
    ],
)

print("Created symphony definition with block referenced twice")
print()


# =============================================================================
# PRINT THE MODEL DUMP
# =============================================================================
print("=" * 60)
print("SYMPHONY STRUCTURE (model_dump)")
print("=" * 60)
print(json.dumps(symphony.model_dump(by_alias=True, mode="json"), indent=2))
print()


# =============================================================================
# DEMONSTRATE THE POWER OF VARIABLES: CHANGE ONE PLACE
# =============================================================================
print("=" * 60)
print("UPDATING THE BLOCK")
print("=" * 60)
print("\nNow let's change GOOG to NVDA - this updates BOTH references!")
print()

# Recreate the blocks with the updated assets
tech_weight_block_updated = WeightCashEqual(
    children=[
        Asset(ticker="MSFT", name="Microsoft Corporation", exchange="XNAS"),
        Asset(ticker="AAPL", name="Apple Inc", exchange="XNAS"),
        Asset(ticker="NVDA", name="NVIDIA Corporation", exchange="XNAS"),  # Changed from GOOG
    ]
)

# Create a new symphony with the updated block
symphony_updated = SymphonyDefinition(
    name="Tech Block Reference Demo - NVDA",
    description="Demonstrates using the same block in multiple places - NVDA version",
    rebalance="daily",
    children=[
        WeightCashEqual(
            children=[
                Group(
                    name="Tech Group 1",
                    collapsed=False,
                    children=[tech_weight_block_updated],
                ),
                Group(
                    name="Tech Group 2",
                    collapsed=False,
                    children=[tech_weight_block_updated],
                ),
            ]
        ),
    ],
)

print("Updated symphony with NVDA instead of GOOG:")
print(json.dumps(symphony_updated.model_dump(by_alias=True, mode="json"), indent=2))
print()


# =============================================================================
# CREATE THE SYMPHONY
# =============================================================================
# To actually create the symphony in your account, uncomment this:
#
result = client.user_symphony.create_symphony(
    name="Tech Block Reference Demo - NVDA",
    color="#829DFF",
    hashtag="#tech",
    symphony=symphony_updated,
)
print(f"Created symphony: {result.symphony_id}")


Created symphony definition with block referenced twice

SYMPHONY STRUCTURE (model_dump)
{
  "id": "8e8a7519-c1ee-4fdf-8f64-43a7b0094107",
  "weight": null,
  "step": "root",
  "name": "Tech Block Reference Demo",
  "description": "Demonstrates using the same block in multiple places",
  "rebalance": "daily",
  "rebalance-corridor-width": null,
  "children": [
    {
      "id": "3dce60ef-0b30-4ab6-8185-5d8d2e9242a5",
      "weight": null,
      "step": "wt-cash-equal",
      "children": [
        {
          "id": "b2370df1-3da3-45e1-9e6d-a2c69af16c28",
          "weight": null,
          "step": "group",
          "name": "Tech Group 1",
          "children": [
            {
              "id": "6013ddbd-aa27-4de9-84c9-663fadf08b0e",
              "weight": null,
              "step": "wt-cash-equal",
              "children": [
                {
                  "id": "4025f368-b583-431f-937c-cb3ee866fb03",
                  "weight": null,
                  "step": "asset",
       